In [5]:
import redelex.datasets
from relbench.datasets import get_dataset

dataset = get_dataset("ctu-basketballmen",download=False)
db = dataset.get_db()

print(db)
print(db.table_dict.keys())

/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Database object from /home/burak/.cache/relbench/ctu-basketballmen/db...
Done in 2.68 seconds.
Database()
dict_keys(['awards_coaches', 'awards_players', 'draft', 'players_teams', 'player_allstar', 'players', 'series_post', 'coaches', 'teams'])


In [6]:
for name, table in db.table_dict.items():
    print(f"\n{'=' * 60}")
    print(name)
    print(table.df.columns.tolist())
    print("shape:", table.df.shape)


awards_coaches
['__PK__', 'year', 'coachID', 'award', 'lgID', 'note', 'FK_coaches_coachID_year']
shape: (61, 7)

awards_players
['__PK__', 'playerID', 'award', 'year', 'lgID', 'note', 'pos', 'FK_players_playerID']
shape: (1719, 8)

draft
['__PK__', 'draftYear', 'draftRound', 'draftSelection', 'draftOverall', 'tmID', 'firstName', 'lastName', 'suffixName', 'playerID', 'draftFrom', 'lgID', 'FK_teams_tmID_draftYear']
shape: (8621, 13)

players_teams
['__PK__', 'playerID', 'year', 'stint', 'tmID', 'lgID', 'GP', 'GS', 'minutes', 'points', 'oRebounds', 'dRebounds', 'rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'PF', 'fgAttempted', 'fgMade', 'ftAttempted', 'ftMade', 'threeAttempted', 'threeMade', 'PostGP', 'PostGS', 'PostMinutes', 'PostPoints', 'PostoRebounds', 'PostdRebounds', 'PostRebounds', 'PostAssists', 'PostSteals', 'PostBlocks', 'PostTurnovers', 'PostPF', 'PostfgAttempted', 'PostfgMade', 'PostftAttempted', 'PostftMade', 'PostthreeAttempted', 'PostthreeMade', 'note', 'FK_playe

In [3]:
for name, table in db.table_dict.items():
    print(f"\n{name}")
    for col in table.df.columns:
        if col.startswith("FK_"):
            print("  ", col)


awards_coaches
   FK_coaches_coachID_year

awards_players
   FK_players_playerID

draft
   FK_teams_tmID_draftYear

players_teams
   FK_players_playerID
   FK_teams_tmID_year

player_allstar
   FK_players_playerID

players

series_post
   FK_teams_tmIDWinner_year
   FK_teams_tmIDLoser_year

coaches
   FK_teams_tmID_year

teams


In [12]:
import featuretools as ft


players = db.table_dict["players"].df.copy()
players_teams=db.table_dict["players_teams"].df.copy()
teams=db.table_dict["teams"].df.copy()
print(players.head(5))
print(players_teams.head(5))



dataframes = {
    "players": (players, "__PK__"),
    "players_teams": (players_teams, "__PK__"),
    "teams": (teams, "__PK__"),
}

relationships = [
    ("players", "__PK__", "players_teams", "FK_players_playerID"),
    ("teams", "__PK__", "players_teams", "FK_teams_tmID_year"),
]


es = ft.EntitySet(id="basketballmen")

for table_name, (df, primary_key) in dataframes.items():
    es = es.add_dataframe(
        dataframe_name=table_name,
        dataframe=df,
        index=primary_key,
    )

print(es)




   __PK__ useFirst firstName middleName      lastName nameGiven  \
0       0     Alaa      Alaa       <NA>     Abdelnaby      <NA>   
1       1   Kareem    Kareem       <NA>  Abdul-Jabbar      <NA>   
2       2    Mahdi     Mahdi       <NA>  Abdul-Rahman      <NA>   
3       3  Mahmoud   Mahmoud       <NA>    Abdul-Rauf      <NA>   
4       4    Tariq     Tariq       <NA>   Abdul-Wahad      <NA>   

                   fullGivenName nameSuffix  nameNick  pos  ...  birthDate  \
0                           <NA>       <NA>      <NA>  F-C  ... 1968-06-24   
1  Ferdinand Lewis Alcindor, Jr.       <NA>  Lew, Cap    C  ... 1947-04-16   
2    Walter Raphael Hazzard, Jr.       <NA>      Walt    G  ... 1942-04-15   
3            Chris Wayne Jackson       <NA>      <NA>    G  ... 1969-03-09   
4     Olivier Michael Saint-Jean       <NA>      <NA>  G-F  ... 1974-11-03   

        birthCity  birthState  birthCountry             highSchool  \
0           Cairo        <NA>           EGY      Bloomfiel

/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40

Entityset: basketballmen
  DataFrames:
    players [Rows: 5062, Columns: 26]
    players_teams [Rows: 23751, Columns: 45]
    teams [Rows: 1536, Columns: 61]
  Relationships:
    No relationships


/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40

In [13]:
for parent_table, parent_col, child_table, child_col in relationships:
    es = es.add_relationship(
        parent_dataframe_name=parent_table,
        parent_column_name=parent_col,
        child_dataframe_name=child_table,
        child_column_name=child_col,
    )

print(es.relationships)

[<Relationship: players_teams.FK_players_playerID -> players.__PK__>, <Relationship: players_teams.FK_teams_tmID_year -> teams.__PK__>]


In [25]:
feature_matrix_players, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="players",
        agg_primitives=[
        "count",
        "sum",
        "mean",
        "min",
        "max",
        "std",
    ],
    trans_primitives=[
        "year",
        "month",
        "day",
    ],
    max_depth=2,
)



/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/featuretools/computational_backends/feature_set_calculator.py:828: FutureWarning: The provided callable <function std at 0x7fb1103dae80> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  ).agg(to_agg)
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/featuretools/computational_backends/feature_set_calculator.py:828: FutureWarning: The provided callable <function mean at 0x7fb1103dad40> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  ).agg(to_agg)
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/featuretools/computational_backends/feature_set_calculator.py:828: FutureWarning: The provided callable <function min at 0x7fb1103da480> is current

In [ ]:
print(feature_matrix_players)

       nameSuffix  pos  firstseason  lastseason  height  weight  \
__PK__                                                            
0             NaN  F-C            0           0    82.0     240   
1             NaN    C            0           0    85.0     225   
2             NaN    G            0           0    74.0     185   
3             NaN    G            0           0    73.0     162   
4             NaN  G-F            0           0    78.0     223   
...           ...  ...          ...         ...     ...     ...   
5057          NaN  G-F            0           0    72.0     185   
5058          NaN    G            0           0    75.0     195   
5059          NaN    C            0           0    85.0     240   
5060          Jr.    G            0           0    73.0     170   
5061          NaN  G-F            0           0    75.0     195   

                  college birthState birthCountry  hsState  ...  \
__PK__                                                      .

In [26]:
feature_matrix_teams, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="teams",
        agg_primitives=[
        "count",
        "sum",
        "mean",
        "min",
        "max",
        "std",
    ],
    trans_primitives=[
        "year",
        "month",
        "day",
    ],
    max_depth=2,
)


/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/featuretools/synthesis/dfs.py:321: UnusedPrimitiveWarning: Some specified primitives were not used during DFS:
  trans_primitives: ['day', 'month', 'year']
This may be caused by a using a value of max_depth that is too small, not setting interesting values, or it may indicate no compatible columns for the primitive were found in the data. If the DFS call contained multiple instances of a primitive in the list above, none of them were used.
  warnings.warn(warning_msg, UnusedPrimitiveWarning)
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/featuretools/computational_backends/feature_set_calculator.py:828: FutureWarning: The provided callable <function mean at 0x7fb1103dad40> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  ).agg(to_agg)
/home/burak/.local/share/mamba/envs/gt/li

In [36]:
print(feature_matrix_teams)

        year lgID tmID franchID confID divID  rank  confRank playoff  \
__PK__                                                                 
0       1937  NBL  AFS      AFS    NaN    EA     1         0      CF   
1       1937  NBL  AGW      AGW    NaN    EA     2         0      WC   
2       1937  NBL  BFB      BFB    NaN    EA     4         0     NaN   
3       1937  NBL  CNC      CNC    NaN    WE     5         0     NaN   
4       1937  NBL  COL      COL    NaN    EA     6         0     NaN   
...      ...  ...  ...      ...    ...   ...   ...       ...     ...   
1531    2011  NBA  SAC      SAC     WC    PC     5         0     NaN   
1532    2011  NBA  SAS      SAS     WC    SW     1         0      CF   
1533    2011  NBA  TOR      TOR     EC    AT     4         0     NaN   
1534    2011  NBA  UTA      UTA     WC    NW     3         0      C1   
1535    2011  NBA  WAS      WAS     EC    SE     4         0     NaN   

                                               name  ...  \
__P

## Implementation 

In [2]:
import redelex.datasets
from relbench.datasets import get_dataset

dataset = get_dataset("rel-f1",download=False)
db = dataset.get_db()

print(db)
print(db.table_dict.keys())

/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Database object from /home/burak/.cache/relbench/rel-f1/db...
Done in 0.52 seconds.
Database()
dict_keys(['constructor_results', 'races', 'results', 'constructors', 'standings', 'circuits', 'qualifying', 'drivers', 'constructor_standings'])


In [7]:
import featuretools as ft
import pandas as pd


# ============================================================
# 1. Bütün tabloları al
# ============================================================
dataframes = {}

for table_name, table in db.table_dict.items():
    print(table.pkey_col)
    print(table.fkey_col_to_pkey_table)



constructorResultsId
{'raceId': 'races', 'constructorId': 'constructors'}
raceId
{'circuitId': 'circuits'}
resultId
{'raceId': 'races', 'driverId': 'drivers', 'constructorId': 'constructors'}
constructorId
{}
driverStandingsId
{'raceId': 'races', 'driverId': 'drivers'}
circuitId
{}
qualifyId
{'raceId': 'races', 'driverId': 'drivers', 'constructorId': 'constructors'}
driverId
{}
constructorStandingsId
{'raceId': 'races', 'constructorId': 'constructors'}


In [5]:
for name, table in db.table_dict.items():
    print(f"\n{'=' * 60}")
    print(name)
    print(table.df.columns.tolist())
    print("shape:", table.df.shape)


constructor_results
['constructorResultsId', 'raceId', 'constructorId', 'points', 'date']
shape: (9408, 5)

races
['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time']
shape: (820, 7)

results
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionOrder', 'points', 'laps', 'milliseconds', 'fastestLap', 'rank', 'statusId', 'date']
shape: (20323, 15)

constructors
['constructorId', 'constructorRef', 'name', 'nationality']
shape: (211, 4)

standings
['driverStandingsId', 'raceId', 'driverId', 'points', 'position', 'wins', 'date']
shape: (28115, 7)

circuits
['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt']
shape: (77, 8)

qualifying
['qualifyId', 'raceId', 'driverId', 'constructorId', 'number', 'position', 'date']
shape: (4082, 7)

drivers
['driverId', 'driverRef', 'code', 'forename', 'surname', 'dob', 'nationality']
shape: (857, 7)

constructor_standings
['constructorStandingsId', 'raceId', 'constructorId', 'po

In [11]:

# ============================================================
# 2. EntitySet oluştur
# ============================================================

es = ft.EntitySet(id="f1")


for table_name, (df, primary_key) in dataframes.items():

    es = es.add_dataframe(
        dataframe_name=table_name,
        dataframe=df,
        index=primary_key,
    )


In [12]:

# ============================================================
# 3. Relationship construction
# ============================================================
def create_featuretools_relationships(db, entityset):
    relationships = []

    for child_table_name, child_table in db.table_dict.items():

        fk_dict = child_table.fkey_col_to_pkey_table

        for fk_col, parent_table_name in fk_dict.items():

            parent_table = db.table_dict[parent_table_name]
            parent_pk = parent_table.pkey_col

            relationship = ft.Relationship(
                entityset=entityset,
                parent_dataframe_name=parent_table_name,
                parent_column_name=parent_pk,
                child_dataframe_name=child_table_name,
                child_column_name=fk_col,
            )

            relationships.append(relationship)

    return relationships



In [13]:

# ============================================================
# 4. Featuretools'a relationship'leri ekle
# ============================================================

def add_relations_to_entityset(db, entityset):

    for child_table_name, child_table in db.table_dict.items():

        for fk_col, parent_table_name in child_table.fkey_col_to_pkey_table.items():

            parent_table = db.table_dict[parent_table_name]
            parent_pk = parent_table.pkey_col

            entityset.add_relationship(
                parent_dataframe_name=parent_table_name,
                parent_column_name=parent_pk,
                child_dataframe_name=child_table_name,
                child_column_name=fk_col,
            )

    return entityset

# ============================================================
# 5. Son kontrol
# ============================================================

# önce dataframe'leri ekliyorsun
for table_name, table in db.table_dict.items():
    es.add_dataframe(
        dataframe_name=table_name,
        dataframe=table.df,
        index=table.pkey_col,
    )

# sonra relations
es = add_relations_to_entityset(db, es)

In [13]:
for relationship in es.relationships:
    print(relationship)

<Relationship: constructor_results.raceId -> races.raceId>
<Relationship: constructor_results.constructorId -> constructors.constructorId>
<Relationship: races.circuitId -> circuits.circuitId>
<Relationship: results.raceId -> races.raceId>
<Relationship: results.driverId -> drivers.driverId>
<Relationship: results.constructorId -> constructors.constructorId>
<Relationship: standings.raceId -> races.raceId>
<Relationship: standings.driverId -> drivers.driverId>
<Relationship: qualifying.raceId -> races.raceId>
<Relationship: qualifying.driverId -> drivers.driverId>
<Relationship: qualifying.constructorId -> constructors.constructorId>
<Relationship: constructor_standings.raceId -> races.raceId>
<Relationship: constructor_standings.constructorId -> constructors.constructorId>


In [14]:
fm, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="drivers",
    agg_primitives=[
        "count",
        "sum",
        "mean",
    ],
    max_depth=3,
)



    

/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/featuretools/computational_backends/feature_set_calculator.py:828: FutureWarning: The provided callable <function mean at 0x7ff2e81da160> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  ).agg(to_agg)
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/featuretools/computational_backends/feature_set_calculator.py:828: FutureWarning: The provided callable <function sum at 0x7ff2e81d8d60> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  ).agg(to_agg)
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/featuretools/computational_backends/feature_set_calculator.py:828: FutureWarning: The provided callable <function sum at 0x7ff2e81d8d60> is current

## Script tries


In [1]:
import redelex.datasets
from relbench.datasets import get_dataset

dataset = get_dataset("rel-f1",download=False)
db = dataset.get_db()

print(db)
print(db.table_dict.keys())

/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Database object from /home/burak/.cache/relbench/rel-f1/db...
Done in 0.48 seconds.
Database()
dict_keys(['constructor_results', 'races', 'results', 'constructors', 'standings', 'circuits', 'qualifying', 'drivers', 'constructor_standings'])


In [5]:
from agg_features import precompute_agg_features

print(precompute_agg_features("rel-f1",db,'drivers',2))

/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/burak/.local/share/mamba/envs/gt/lib/python3.12/site-packages/woodwork/type_sys/utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/burak/.local/share/mamba/envs/gt/li

(         code nationality  COUNT(results)  MAX(results.fastestLap)  \
driverId                                                             
0         HAM     British              52                     71.0   
1         HEI      German             168                     72.0   
2         ROS      German              70                     70.0   
3         ALO     Spanish             140                     73.0   
4         KOV     Finnish              52                     74.0   
...       ...         ...             ...                      ...   
852       MSC      German               0                      NaN   
853       ZHO     Chinese               0                      NaN   
854       DEV       Dutch               0                      NaN   
855       PIA  Australian               0                      NaN   
856       SAR    American               0                      NaN   

          MAX(results.grid)  MAX(results.laps)  MAX(results.milliseconds)  \
driverId   